# VulnSage Phase 3D — Kaggle Pilot v2
Canonical positive-only pilot for `qwen2.5-coder:7b` through local Ollama. This notebook fails closed until an exact reviewed Git commit, an official READY-only manifest, model availability, context preflight, and explicit inference confirmation are all present. It never reads the legacy CSV CLI.

In [ ]:
# REVIEW AND EDIT THIS CELL BEFORE ANY RUN. Do not put secrets here.
REPOSITORY_URL = 'https://github.com/TranPhuocToan/vulnsage-pipeline.git'
APPROVED_GIT_COMMIT = 'REPLACE_WITH_REVIEWED_40_HEX_COMMIT'
MODEL_ID = 'qwen2.5-coder:7b'
OFFICIAL_PACKAGE_DIR = '/kaggle/input/vulnsage-approved-pilot'
TOKENIZER_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
RUN_ID = 'phase3d-pilot-REPLACE_WITH_UNIQUE_RUN_ID'
NUM_CTX = 32768
NUM_PREDICT = 2048
SEED = 25550029
MAX_ATTEMPTS = 3
ALLOW_OLLAMA_INSTALL = False
ALLOW_MODEL_DOWNLOAD = False
ALLOW_TOKENIZER_INSTALL = False
ALLOW_TOKENIZER_DOWNLOAD = False
CONFIRM_PAID_OR_GPU_INFERENCE = False


In [ ]:
# Hardware and immutable configuration precheck — no model call.
import os, re, shutil, subprocess, sys
from pathlib import Path
assert re.fullmatch(r'[0-9a-f]{40}', APPROVED_GIT_COMMIT), 'Set the reviewed full Git SHA; branch/latest is forbidden.'
assert re.fullmatch(r'[A-Za-z0-9._-]+', RUN_ID) and 'REPLACE' not in RUN_ID
print('Python:', sys.version)
print('Disk:', shutil.disk_usage('/kaggle/working'))
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], text=True, capture_output=True)
assert gpu.returncode == 0, 'GPU is required for the planned smoke test/pilot.'
print(gpu.stdout)


In [ ]:
# Clone/fetch exactly one approved commit; never pull a moving branch.
REPO_DIR = Path('/kaggle/working/vulnsage-pipeline')
assert not REPO_DIR.exists(), f'Refusing to reuse existing directory: {REPO_DIR}'
subprocess.run(['git', 'init', str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', APPROVED_GIT_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == APPROVED_GIT_COMMIT
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'dataset_validation/requirements.txt')], check=True)
sys.path.insert(0, str(REPO_DIR))


In [ ]:
# Official artifact and no-LLM context preflight.
from pipeline.context_policy import ContextPolicy
from pipeline.pilot_runner import validate_official_package_inputs
try:
    import transformers
    from transformers import AutoTokenizer
except ImportError:
    assert ALLOW_TOKENIZER_INSTALL, 'Review then set ALLOW_TOKENIZER_INSTALL=True.'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1'], check=True)
    import transformers
    from transformers import AutoTokenizer
PACKAGE_DIR = Path(OFFICIAL_PACKAGE_DIR)
SCHEMA_PATH = REPO_DIR / 'dataset_validation/schemas/normalized_sample.schema.json'
assert PACKAGE_DIR.is_dir(), 'The complete five-file official package is required.'
context_policy = ContextPolicy(max_utf8_bytes_per_call=2_000_000)
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, local_files_only=not ALLOW_TOKENIZER_DOWNLOAD, trust_remote_code=False)
def exact_token_counter(text): return len(tokenizer.encode(text, add_special_tokens=False))
tokenizer_metadata = {'id': TOKENIZER_ID, 'library': 'transformers', 'version': transformers.__version__, 'class': type(tokenizer).__name__, 'scope': 'rendered_raw_prompt_without_ollama_template'}
dry_run = validate_official_package_inputs(package_dir=PACKAGE_DIR, schema_path=SCHEMA_PATH, expected_git_commit=APPROVED_GIT_COMMIT, context_policy=context_policy, model_context_tokens=NUM_CTX, reserved_output_tokens=NUM_PREDICT, exact_token_counter=exact_token_counter)
print(dry_run)
assert dry_run['status'] == 'PASS_NO_LLM', 'Context strategy requires approval; do not truncate or alter prompts ad hoc.'
assert dry_run['exact_token_preflight_status'] == 'PASS_STATIC_PROMPTS_RUNTIME_GUARD_REQUIRED', 'Exact tokenizer preflight must pass for all static prompts.'


In [ ]:
# Install/start local Ollama only when explicitly enabled. No API key is used.
if shutil.which('ollama') is None:
    assert ALLOW_OLLAMA_INSTALL, 'Review then set ALLOW_OLLAMA_INSTALL=True.'
    subprocess.run(['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
OLLAMA_LOG = open('/kaggle/working/ollama-server.log', 'ab')
ollama_server = subprocess.Popen(['ollama', 'serve'], stdout=OLLAMA_LOG, stderr=subprocess.STDOUT)
import time
time.sleep(5)
from pipeline.ollama_transport import OllamaConfig, OllamaGenerateTransport
ollama_config = OllamaConfig(model=MODEL_ID, num_ctx=NUM_CTX, num_predict=NUM_PREDICT, seed=SEED)
probe_transport = OllamaGenerateTransport(ollama_config)
model_metadata = probe_transport.probe()
if not model_metadata['available']:
    assert ALLOW_MODEL_DOWNLOAD, 'Review disk/network cost then set ALLOW_MODEL_DOWNLOAD=True.'
    subprocess.run(['ollama', 'pull', MODEL_ID], check=True)
    model_metadata = probe_transport.probe()
assert model_metadata['available'] and model_metadata['digest']
print(model_metadata)


## Explicit inference gate
Stop here and inspect: exact Git SHA, official manifest/hash, READY-only sample list, dry-run context status, GPU/RAM/disk, Ollama model name/digest, and output run ID. Only then change `CONFIRM_PAID_OR_GPU_INFERENCE` to `True` and rerun from the configuration cell.

In [ ]:
# Canonical P0/P1/P2 pilot. This is the first cell allowed to invoke the model.
assert CONFIRM_PAID_OR_GPU_INFERENCE is True, 'Explicit researcher confirmation required.'
import json
from pipeline.llm_client import StructuredLLMClient
from pipeline.pilot_runner import run_approved_pilot
OUTPUT_ROOT = Path('/kaggle/working/vulnsage-runs') / RUN_ID
assert not OUTPUT_ROOT.exists(), 'Choose a new RUN_ID; historical outputs are immutable.'
OUTPUT_ROOT.mkdir(parents=True)
transports = {}
def client_factory(pipeline):
    transport = OllamaGenerateTransport(ollama_config)
    transports[pipeline] = transport
    return StructuredLLMClient(transport, max_attempts=MAX_ATTEMPTS, retry_delays=[1.0, 2.0, 4.0])
model_quantization = model_metadata.get('details', {}).get('quantization_level')
assert model_quantization, 'Ollama model quantization metadata is required.'
summary = run_approved_pilot(package_dir=PACKAGE_DIR, schema_path=SCHEMA_PATH, output_root=OUTPUT_ROOT, run_prefix=RUN_ID, git_commit=APPROVED_GIT_COMMIT, model_id=MODEL_ID, model_digest=model_metadata['digest'], model_quantization=model_quantization, tokenizer_metadata=tokenizer_metadata, parameters=ollama_config.run_parameters(), retry_policy={'max_attempts': MAX_ATTEMPTS, 'retry_delays_sec': [1.0, 2.0, 4.0]}, client_factory=client_factory, context_policy=context_policy, model_context_tokens=NUM_CTX, reserved_output_tokens=NUM_PREDICT, exact_token_counter=exact_token_counter)
with (OUTPUT_ROOT / 'pilot_summary.json').open('x', encoding='utf-8') as f: json.dump(summary, f, indent=2, sort_keys=True)
with (OUTPUT_ROOT / 'model_metadata.json').open('x', encoding='utf-8') as f: json.dump(model_metadata, f, indent=2, sort_keys=True)
with (OUTPUT_ROOT / 'ollama_call_log.json').open('x', encoding='utf-8') as f: json.dump({k: v.call_log for k, v in transports.items()}, f, indent=2, sort_keys=True)
print(summary)


In [ ]:
# Export all run-scoped artifacts for download.
assert summary['complete'], 'Do not describe an interrupted run as complete; resume with the same immutable spec.'
archive = shutil.make_archive(str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT)
print('Download artifact:', archive)
